In [36]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.preprocessing import MaxAbsScaler

attributes = pd.read_csv('../data/fit_frenchie_attributes.csv')
attributes = attributes.astype(str)
attributes['nft_id'] = attributes['nft_id'].astype(int)

trades = pd.read_csv('../data/fit_frenchie_trades.csv')
trades = trades.astype(float)
trades['nft_id'] = trades['nft_id'].astype(int)

nft_1046 = pd.read_csv('../data/fit_frenchie_1046_attributes_freq.csv')

df = pd.read_csv('../data/fit_frenchie_trades_attributes_freq.csv')
df['normalized_trade_price'] = df['trade_price'] / df['floor_price']


basic_features = ['Accessories', 'Background', 'Body', 'Eyes', 'Hat', 'Mouth']
features = basic_features + [col for col in df.columns if col.endswith('_freq')]
X = df[features]
y = df['normalized_trade_price']  

preprocessor = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=True)),
    ('scaler', MaxAbsScaler())  
])

df

,nft_id,trade_price,floor_price,Accessories,Background,Body,Eyes,Hat,Mouth,Accessories_Background_freq,...,Eyes_Hat_freq,Eyes_Mouth_freq,Hat_Mouth_freq,Mouth_freq,Accessories_freq,Background_freq,Body_freq,Eyes_freq,Hat_freq,normalized_trade_price
0,1088,4.00,4.00,Tie,Midnight Sapphire Blue,Lilac,White Round Shades,No Hat,Sad,0.0073,...,0.0327,0.0120,0.0387,0.1180,0.0627,0.1167,0.0433,0.0987,0.3147,1.000000
1,1088,5.16,5.16,Tie,Midnight Sapphire Blue,Lilac,White Round Shades,No Hat,Sad,0.0073,...,0.0327,0.0120,0.0387,0.1180,0.0627,0.1167,0.0433,0.0987,0.3147,1.000000
2,1088,8.15,8.15,Tie,Midnight Sapphire Blue,Lilac,White Round Shades,No Hat,Sad,0.0073,...,0.0327,0.0120,0.0387,0.1180,0.0627,0.1167,0.0433,0.0987,0.3147,1.000000
3,1088,4.14,4.14,Tie,Midnight Sapphire Blue,Lilac,White Round Shades,No Hat,Sad,0.0073,...,0.0327,0.0120,0.0387,0.1180,0.0627,0.1167,0.0433,0.0987,0.3147,1.000000
4,1065,4.00,4.00,NaN,Spiced Orange,Tan,White Shades,Headphones,Toothy,0.0527,...,0.0087,0.0300,0.0087,0.1853,0.4247,0.1180,0.1553,0.1480,0.0500,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,36,1.51,0.95,Wings,Lavender Dusk Purple,Tan,Laser Eyes,No Hat,Sad,0.0013,...,0.0060,0.0013,0.0387,0.1180,0.0193,0.1187,0.1553,0.0167,0.3147,1.589474
1996,1028,0.94,0.94,Red Bandana,Forest Moss Green,Black - Overweight,Red Round Shades,Cylinder,Toothy,0.0167,...,0.0060,0.0127,0.0107,0.1853,0.0920,0.1247,0.1773,0.0587,0.0493,1.000000
1997,59,0.94,0.94,NaN,Sunlit Gold,Tan,Drowsy,Party Hat,Drooling,0.0673,...,0.0027,0.0053,0.0040,0.0700,0.4247,0.1500,0.1553,0.0407,0.0500,1.000000
1998,614,0.91,0.91,Neck Tattoo,Lavender Dusk Purple,Black,White Round Shades,Slickback,Toothy,0.0047,...,0.0033,0.0147,0.0107,0.1853,0.0300,0.1187,0.2333,0.0987,0.0493,1.000000


In [39]:
X_processed = preprocessor.fit_transform(X)

num_clusters = 50

kmeans = KMeans(n_clusters=num_clusters, random_state=42)
clusters = kmeans.fit_predict(X_processed)

df['cluster'] = clusters

cluster_avg_prices = df.groupby('cluster')['normalized_trade_price'].mean().reset_index()
cluster_avg_prices.columns = ['cluster', 'average_price']

cluster_avg_prices.set_index('cluster').round(2)

,average_price
cluster,
0,1.26
1,1.26
2,1.02
3,1.02
4,1.12
5,1.71
6,1.17
7,1.47
8,1.43


In [40]:
nft_1046 = pd.concat([df.drop(['trade_price'], axis=1),nft_1046])[features].tail(1)

nft_1046_processed = preprocessor.transform(nft_1046)

nft_1046_cluster = kmeans.predict(nft_1046_processed)

predicted_price = cluster_avg_prices.loc[cluster_avg_prices['cluster'] == nft_1046_cluster[0], 'average_price'].values[0]

print(f"Predicted Price for the new NFT: {predicted_price:.2f}")

Predicted Price for the new NFT: 1.02


In [41]:
nn_model = NearestNeighbors(n_neighbors=6)
nn_model.fit(X_processed)

NearestNeighbors(n_neighbors=6)

In [42]:
new_nft_processed = preprocessor.transform(nft_1046)
distances, indices = nn_model.kneighbors(new_nft_processed)
neighbor_indices = indices[0][1:]
neighbor_prices = df.iloc[neighbor_indices]['normalized_trade_price']
predicted_price = neighbor_prices.mean()

print(f"Predicted Price for the new NFT: {predicted_price:.2f}")

Predicted Price for the new NFT: 1.01


In [43]:
df.iloc[neighbor_indices]

,nft_id,trade_price,floor_price,Accessories,Background,Body,Eyes,Hat,Mouth,Accessories_Background_freq,...,Eyes_Mouth_freq,Hat_Mouth_freq,Mouth_freq,Accessories_freq,Background_freq,Body_freq,Eyes_freq,Hat_freq,normalized_trade_price,cluster
1047,1326,7.01,6.55,Bow Tie,Rose Quartz Pink,Tan,Eye Patch,No Hat,Bubble Gum,0.0093,...,0.0053,0.0107,0.0500,0.0933,0.1160,0.1553,0.08,0.3147,1.070229,48
483,853,5.49,5.49,Bow Tie,Stormcloud Grey,Tan,Eye Patch,No Hat,Surprised,0.0120,...,0.0093,0.0407,0.1187,0.0933,0.1187,0.1553,0.08,0.3147,1.000000,48
484,853,6.74,6.74,Bow Tie,Stormcloud Grey,Tan,Eye Patch,No Hat,Surprised,0.0120,...,0.0093,0.0407,0.1187,0.0933,0.1187,0.1553,0.08,0.3147,1.000000,48
485,853,7.59,7.59,Bow Tie,Stormcloud Grey,Tan,Eye Patch,No Hat,Surprised,0.0120,...,0.0093,0.0407,0.1187,0.0933,0.1187,0.1553,0.08,0.3147,1.000000,48
795,692,6.77,6.77,NaN,Rose Quartz Pink,Tan,Eye Patch,No Hat,Feral,0.0480,...,0.0007,0.0153,0.0480,0.4247,0.1160,0.1553,0.08,0.3147,1.000000,26


In [26]:
accessor_1046   = nft_1046[basic_features].Accessories.values[0]
background_1046 = nft_1046[basic_features].Background.values[0]
body_1046       = nft_1046[basic_features].Body.values[0]
eyes_1046       = nft_1046[basic_features].Eyes.values[0]
hat_1046        = nft_1046[basic_features].Hat.values[0]
mouth_1046      = nft_1046[basic_features].Mouth.values[0]

df[
    (df.Accessories == accessor_1046) &
    # (df.Background == background_1046) &
    # (df.Body == body_1046) &
    # (df.Eyes == eyes_1046) &
    # (df.Hat == hat_1046) &
    (df.Mouth == mouth_1046)
].trade_price.mean()

12.7025

In [77]:
import itertools

features = ['Accessories', 'Background', 'Body', 'Eyes', 'Hat', 'Mouth']
attribute_combinations = []

for i in range(1, len(features) + 1):
    attribute_combinations.extend(itertools.combinations(features, i))

results = []

for combination in attribute_combinations:
    filter_conditions = {attr: nft_1046[attr].values[0] for attr in combination}
    
    filtered_df = df.copy()
    for attr, value in filter_conditions.items():
        filtered_df = filtered_df[filtered_df[attr] == value]
    
    if not filtered_df.empty:
        average_price = filtered_df['normalized_trade_price'].mean()
        freq_col = '_'.join(f"{attr}" for attr in filter_conditions.keys()) + '_freq'
        try:
            freq = nft_1046[freq_col].values[0]
        except:
            freq = 0
        results.append({
            'Attributes': ', '.join(f"{attr}" for attr, value in filter_conditions.items()),
            'Normalized Price': average_price,
            'Number of NFTs': len(filtered_df),
            'Frequency': freq
        })

results_df = pd.DataFrame(results)
# scale results_df.Frequency to have the reverse range of 0-1
results_df['scaled_freq'] = 1 - (results_df.Frequency - results_df.Frequency.min()) / (results_df.Frequency.max() - results_df.Frequency.min())
results_df['w_scaled_freq'] = results_df['scaled_freq']/results_df['scaled_freq'].sum()
results_df.round(3)

,Attributes,Normalized Price,Number of NFTs,Frequency,scaled_freq,w_scaled_freq
0,Accessories,1.537,38,0.093,0.704,0.037
1,Background,1.319,220,0.116,0.631,0.033
2,Body,1.246,265,0.155,0.507,0.027
3,Eyes,1.336,52,0.080,0.746,0.040
4,Hat,2.179,25,0.315,0.000,0.000
5,Mouth,1.269,239,0.050,0.841,0.045
6,"Accessories, Background",1.555,6,0.009,0.970,0.051
7,"Accessories, Body",1.371,8,0.018,0.943,0.050
8,"Accessories, Mouth",1.650,4,0.004,0.987,0.052
9,"Background, Body",1.279,29,0.018,0.943,0.050


In [75]:
final_weighted_price = (results_df['Normalized Price'] * results_df['w_scaled_freq']).sum()
recent_floor_price = df['floor_price'].tail(5).mean()
print(f"Final weighted normalized price: {final_weighted_price:.2f}")
print(f"Most recent floor price: {recent_floor_price:.2f}")
print(f"Most recent trade price: {(recent_floor_price*final_weighted_price):.2f}")

Final weighted normalized price: 1.46
Most recent floor price: 0.93
Most recent trade price: 1.36


In [70]:
df[['trade_price', 'floor_price']].tail(20)

,trade_price,floor_price
1980,1.76,1.70
1981,1.70,1.70
1982,1.70,1.70
1983,2.32,1.70
1984,2.73,1.70
1985,2.03,1.70
1986,4.97,1.75
1987,2.43,1.80
1988,1.80,1.80
1989,1.60,1.60
